In [ ]:
import pyvisa
import logging
import re

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

class SRSFreqSys:
    def __init__(self, resource: str, time_out: int=5000):
        rm = pyvisa.ResourceManager()
        self._instr = rm.open_resource(resource)
        self._instr.timeout = time_out
    
    def __enter__(self):
        return self
    
    def __exit__(self, exc_type, exc, tb):
        self._instr.close()

    def _write(self, cmd: str, time_out: int|None=None) -> None:
        old_timeout = self._instr.timeout
        if time_out is not None:
            self._instr.timeout = time_out
            logger.debug("Set timeout to %s from %s for command %s",
                         time_out, old_timeout, cmd)
        try:
            logger.debug("Writing command: %s", cmd)
            self._instr.write(cmd)
        finally:
            self._instr.timeout = old_timeout

    def _query(self, cmd: str, time_out: int|None=None) -> str:
        old_timeout = self._instr.timeout
        if time_out is not None:
            logger.debug("Set timeout to %s from %s for command %s",
                         time_out, old_timeout, cmd)
            self._instr.timeout = time_out
        try:
            logger.debug("Querying command: %s", cmd)
            return self._instr.query(cmd)
        finally:
            self._instr.timeout = old_timeout

    def ModOnOff(self, QoS: str, switch: str|None=None) -> str:
        """Query or Set the current status of modulation.
        QoS: Query or Set indicator. Q: Query. S: Set."""
        if QoS.lower() == "q":
            resp = self._query("MODL ?")
            # SRS SG380 series usually returns '0' or '1' as strings
            return "Mod On" if resp == "1" else "Mod Off"
        
        elif QoS.lower() == "s" and switch is not None:
            if switch.lower() == "on":
                self._write("MODL 1")
                return "Mod turned on"
            elif switch.lower() == "off":
                self._write("MODL 0")
                return "Mod turned off"
            else:
                logger.error("Invalid input on On/Off switch")
                raise ValueError("Invalid type of input! Must be On/Off.")
        
        else:
            logger.error("Invalid input on Query/Set")
            raise ValueError("Invalid input! Must be Q/S.")
        
    def CenterFreq(self, QoS: str, freq: str | None = None) -> str:
        """Query or Set the center frequency."""
        
        if QoS.lower() == "q":
            # Requesting specific units from the SRS is smart. 
            # .strip() ensures no trailing whitespace or newlines.
            resp = self._query("FREQ? MHz").strip()
            return f"Center Freq = {resp} MHz"

        elif QoS.lower() == 's' and freq is not None:
            # 1. Clean the input and split numbers from units
            # This regex finds the number part and the text part regardless of spaces
            match = re.search(r"(\d*\.?\d+)\s*([a-zA-Z]*)", freq)
            if not match:
                raise ValueError(f"Could not parse frequency input: {freq}")
            
            num_str, unit_str = match.groups()
            num = float(num_str)
            
            # 2. Safety Check (Assuming SG384 limit is ~4.05 GHz)
            # We normalize to Hz for a consistent safety check
            multiplier = 1
            u = unit_str.lower()
            if 'g' in u: multiplier = 1e9
            elif 'm' in u: multiplier = 1e6
            elif 'k' in u: multiplier = 1e3
            
            total_hz = num * multiplier
            
            if 0 < total_hz <= 4.05e9:
                # 3. Format strictly: SRS wants "FREQ 1.234 GHz" 
                # We use f-string with :f to prevent scientific notation (e.g., 1e-3)
                # but usually, for these instruments, 6-10 decimal places is plenty.
                cmd = f"FREQ {num:.6f} {unit_str.strip()}"
                self._write(cmd)
                return f"Center Freq set to {num:.6f} {unit_str.strip()}"
            else:
                logger.error("Frequency %s is out of range (0 - 4.05 GHz)", freq)
                raise ValueError("Frequency out of hardware range.")

        else:
            logger.error("Invalid input on Query/Set or missing frequency")
            raise ValueError("Invalid input! Must be Q/S and freq must be provided for Set.")

    def RFAmplitude(self, QoS: str, ampl: str | None = None) -> str:
        """Query or Set the RF amplitude (Type N). Note that unit is either dBm or V_RMS."""
        if QoS.lower() == "q":
            resp = self._query("AMPR?").strip()
            return f"RF Amplitude is {resp}"

        elif QoS.lower() == "s" and ampl is not None:
            match = re.search(r"([-+]?\d*\.?\d+)\s*([a-zA-Z]*)", ampl)
            if not match:
                logger.error("Invalid Input! Amplitude must be in unit of dBm or RMS.")
                raise ValueError("Invalid Input! Amplitude must be in unit of dBm or RMS.")

            num_str, unit_str = match.group(1), match.group(2)
            num = float(num_str)

            if unit_str.lower() == "dbm":
                cmd = f"AMPR {num_str.strip()}"
            elif unit_str.lower() == "rms":
                if num > 0:
                    cmd = f"AMPR {num_str} RMS"
                else:
                    logger.error("Invalid input! V_RMS must greater than 0.")
                    raise ValueError("Invalid input! V_RMS must greater than 0.")
            else:
                logger.error("Invalid unit! Unit must be dBm or RMS.")
                raise ValueError("Invalid unit! Unit must be dBm or RMS.")

            self._write(cmd)
            return f"RF amplitude set to {num:.3f} {unit_str}."

        else:
            logger.error("Invalid input on Query/Set or missing amplitude")
            raise ValueError("Invalid input! Must be Q/S and amplitude must be provided for Set.")